# CA 4.2: Brain Tumor Segmentation with U-Net

## Problem definition: Segmentation of gliomas in MRI scans

In this jupyter notebook, we will be doing whole tumor (WT) segmentation. WT includes subregions 1, 2 and 4.



Each pixel on image is labelled as :

- Pixel is part of a tumor area (subregions: 1, 2, 4)
- Pixel is not a part of tumor area (0)

where, 0 - no tumor, 1 - necrotic/core/ non-enhancing, 2 - edema, 4 - enhancing


**Objectivies (What will we learn here?)**
1. Setting up the env
2. Create training, validation and test ids
3. Image pre-processing
4. Performance metrics for semantic segmentation
5. Loss functions for semantic segmentation
6. Building the Unet model
7. Training the model
8. Looking at the learning curve
9. Predicting using the Unet model
10. What next? It is time to play!




## Step 1: Set up the env

Download all the packages needed.

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from tqdm import tqdm

## Step 2: Now let's create the training, validation and test ids.


In [ ]:
# Define the path to your original dataset
DATA_PATH = Path('/courses/CS7150.202630/shared/brats20/BraTS2020_TrainingData')

TRAIN_DATA_PATH = Path(DATA_PATH, 'MICCAI_BraTS2020_TrainingData')

In [3]:
# Let's create the training, validation and test ids.

train_dir = [f.path for f in os.scandir(TRAIN_DATA_PATH) if f.is_dir()]

# We will create train, validation and test ids from the train_dir dataset


def list_to_ids(dir:str):
    """
    Will convert the dir paths to ids by parsing the paths.
    dir: string, image dir paths in BRATS
    """
    x = []
    for i in range(0,len(dir)):
        x.append(dir[i].split('/')[-1])
    return x

How to do train-test-split using sklearn: https://www.stackvidhya.com/train-test-split-using-sklearn-in-python/

In [ ]:
# Now let's use the defined function

ids = list_to_ids(train_dir)

# Split dataset to create training ids, validation ids and test ids
# Here we have selected the size of test set as 20% which is a common practice.
train_ids, test_ids = train_test_split(ids,test_size=0.2)

# Create validation ids by further splitting the train ids, we again use 20% as size of validation set.
# Validation set is also referred to as tuning set.

train_ids, val_ids = train_test_split(train_ids,test_size=0.2)

In [5]:
# Now looks at the number of patient ids in training, validation and test sets
print(f'There are {len(train_ids)} patient ids in training set')
print(f'There are {len(val_ids)} patient ids in validation set')
print(f'There are {len(test_ids)} patient ids in test set')

There are 236 patient ids in training set
There are 59 patient ids in validation set
There are 74 patient ids in test set


### Understanding the Data Split

We use a common 64-16-20 split resulting in:
- **Training set (236 patients)**: Used to learn the model parameters through backpropagation
- **Validation set (59 patients)**: Used to tune hyperparameters and monitor training progress
- **Test set (74 patients)**: Held out completely - the model never sees this during training

This three-way split helps us:
1. **Train** without overfitting (we monitor validation loss)
2. **Validate** hyperparameter choices without "peeking" at test set
3. **Test** final performance on truly unseen data for an unbiased evaluation

This is standard practice in machine learning to ensure models generalize well to new data.

## Step 3: Let's do some image pre-processing. You have already learnt basics of reading and visualizing nifti images, selecting slices and looking at histograms.

Here, we will look at some pre-processing steps specific to semantic segmentation. It's important to understand that the pro-processing steps could vary depending on the task at hand.


In [ ]:
# Let's rewind and remember the type of images available: T1, T1CE, T2, FLAIR and corresponding masks

# We will use the read_patient_niftis module we had learnt in the previous notebook
def read_patient_niftis(patient_id: str,
                        niftis_to_load = ['t1', 't1ce', 't2', 'flair', 'seg'],
                        data_path = TRAIN_DATA_PATH):
    """
    Will read in the images from a single patient and return a dictionary of
    those images with the key as the image type and the value as the nifti object.
    patient_id: string of format '001' through '369', patient ID in BRATS
    niftis_to_load: default is list containing all the types of images that we care about; can sub in another list if desired.
    data_path: default is DATA_PATH specified above with respect to the mounted google drive (in colab)

    Args:
        patient_id (str): Patient ID in BRATS (e.g., 'BraTS20_Training_001').
        niftis_to_load (list): List of image types to load (e.g., ['t1', 't1ce']).
        data_path (str or Path): Path to the dataset.

    Returns:
        dict: Dictionary with keys as image types and values as nifti objects.
    """
    patient_image_dict = {}
    for image in niftis_to_load:
        nii_gz_path = Path(data_path, f'{patient_id}', f'{patient_id}_{image}.nii.gz')
        nii_path = Path(data_path, f'{patient_id}', f'{patient_id}_{image}.nii')

        # Check for the existence of .nii.gz or .nii files
        if nii_gz_path.exists():
            patient_image_dict[image] = nib.load(nii_gz_path)
        elif nii_path.exists():
            patient_image_dict[image] = nib.load(nii_path)
        else:
            raise FileNotFoundError(f"Neither {nii_gz_path} nor {nii_path} found for {patient_id}")
    return patient_image_dict

In [ ]:
# Let's take a look one of the image and the corresponding segmentation mask from the training set before we proceed
pt_img_dict = read_patient_niftis(train_ids[12])

# Now lets visualize all the images side by side to get a sense of what's happening.
plt.figure(figsize=(45,30)) # specifying the overall grid size
for i, (key, value) in enumerate(pt_img_dict.items()):
    plt.subplot(1, 5, i+1)
    plt.imshow(value._dataobj[:, :, value.shape[-1]//2].T, cmap='gray') # Looks halfway through the volume and transposes the image so that its facing upward.
    plt.axis('off')
    plt.title(key, fontsize=30)
plt.show()

A quick recap on MRI basics: https://my-ms.org/mri_basics.html

In [ ]:
# For this semantic segmentation task, we will use T2 images.
# And we will focus on binary or whole tumor (WT) segmentation. In order to do so we will assign the same pixel intensity (1) to the 3 tumor sub regions (1, 2, 4)

def normalize(input_image, input_mask, percentile=0.001, eps=1e-6):
    """
    Normalize the input image between 0 and 1 and prepare the mask for binary segmentation.
    
    Args:
        input_image: The image to be segmented
        input_mask: The ground truth or segmentation labels
        percentile: Percentile for robust normalization (removes outliers)
        eps: Small epsilon to avoid division by zero

    Returns:
        img_normalized: Normalized image as torch tensor in range [0, 1]
        mask_normalized: Binary mask (0 or 1) as torch tensor
    """
    # Convert input image to a PyTorch tensor if it's not already
    if not isinstance(input_image, torch.Tensor):
        input_image = torch.tensor(input_image, dtype=torch.float32)

    # Ensure the tensor is contiguous before reshaping
    img_array = input_image.contiguous().view(-1)

    # Calculate the min and max based on the percentiles
    min_img = torch.quantile(img_array, percentile)
    max_img = torch.quantile(img_array, 1 - percentile)

    # Normalize the image
    img_normalized = (input_image - min_img) / (max_img - min_img + eps)

    # Ensure the image is in the range [0, 1]
    img_normalized = torch.clamp(img_normalized, 0, 1)

    # Convert image to float tensor if not already
    img_normalized = img_normalized.float()

    # Process the mask: convert all tumor regions (1, 2, 4) to 1, background stays 0
    if not isinstance(input_mask, torch.Tensor):
        input_mask = torch.tensor(input_mask, dtype=torch.int32)
    mask_normalized = torch.where(input_mask >= 1, 1, 0)

    return img_normalized, mask_normalized

In [9]:
# Example usage
# Assuming pt_img_dict['t2'] and pt_img_dict['seg'] are loaded as PyTorch tensors
t2_img = torch.tensor(pt_img_dict['t2'].get_fdata(), dtype=torch.float32)
seg_img = torch.tensor(pt_img_dict['seg'].get_fdata(), dtype=torch.int32)

img_normalized, mask_normalized = normalize(t2_img, seg_img)

In [ ]:
# Convert PyTorch tensors to NumPy arrays for plotting
img_np = img_normalized.cpu().numpy()
mask_np = mask_normalized.cpu().numpy()

# Visualize the images
plt.figure(figsize=(20, 30))

# Original T2 image
plt.subplot(1, 3, 1)
plt.imshow(img_np[:, :, 78].T, cmap='gray')  # Assuming the third dimension is the slice axis
plt.axis('off')
plt.title('T2', fontsize=20)

# Original segmentation mask
plt.subplot(1, 3, 2)
plt.imshow(seg_img[:, :, 78].T, cmap='gray')  # Directly using seg_img if it's already a NumPy array
plt.axis('off')
plt.title('Mask of Sub Regions', fontsize=20)

# Normalized mask
plt.subplot(1, 3, 3)
plt.imshow(mask_np[:, :, 78].T, cmap='gray')
plt.axis('off')
plt.title('Mask of Whole Tumor', fontsize=20)

plt.show()

### Why Normalize MRI Images?

Different MRI sequences (T1, T2, FLAIR) have vastly different intensity ranges:
- **T1**: 0-700
- **T2**: 0-300  
- **FLAIR**: 0-600

Neural networks train best when inputs are in a consistent range (typically [0,1]). Without normalization:
- Large intensity values can cause exploding gradients
- Different sequences would have different "importance" just due to scale
- Training becomes unstable and slow

We use **percentile-based normalization** (0.1% to 99.9%) rather than simple min-max to handle outliers robustly.

</br>

---
## Preprocessing for Efficient Training

Before training, we'll preprocess all our data once and store it in memory as numpy arrays. This approach:

**Why preprocess?**
- Loading NIfTI files during training is slow (I/O bottleneck)
- Normalizing volumes repeatedly wastes computation
- GPU sits idle waiting for data

**Our approach:**
1. Load each patient's volume **once**
2. Extract the middle slice (where tumors are most visible)
3. Normalize the 2D slice
4. Store all slices as numpy arrays in memory

This way, during training, we just index into pre-loaded arrays - extremely fast! The preprocessing takes 2-3 minutes but makes each epoch 10x faster.

In [ ]:
def create_numpy_arrays(patient_ids, 
                        modalities=['t2'],
                        slice_indices=[lambda d: d//2],
                        data_path=TRAIN_DATA_PATH):
    """
    Preprocess patient data into numpy arrays for efficient training.
    
    This function:
    1. Loads 3D NIfTI volumes for specified patients
    2. Extracts specific slices
    3. Normalizes each slice
    4. Returns as numpy arrays ready for training
    
    Args:
        patient_ids: List of patient IDs (e.g., ['BraTS20_Training_001', ...])
        modalities: List of MRI modalities to use as input channels (e.g., ['t2'])
        slice_indices: Which slices to extract as functions of depth
                      Default: [lambda d: d//2] (middle slice)
        data_path: Path to BraTS training data
    
    Returns:
        X: numpy array of shape (N, H, W, C) where C = len(modalities)
        Y: numpy array of shape (N, H, W, 1) containing binary masks
    """
    all_images = []
    all_masks = []
    
    print(f"Preprocessing {len(patient_ids)} patients with {len(modalities)} modality(ies)...")
    
    for patient_id in tqdm(patient_ids, desc="Loading patients"):
        # Load NIfTI volumes for this patient
        niftis_to_load = modalities + ['seg']
        pt_img_dict = read_patient_niftis(patient_id, 
                                          niftis_to_load=niftis_to_load,
                                          data_path=data_path)
        
        # Get volume depth (same for all modalities)
        depth = pt_img_dict['seg'].shape[2]
        
        # Extract each requested slice
        for slice_fn in slice_indices:
            slice_idx = slice_fn(depth)
            
            # Normalize each modality for this slice
            channels = []
            for modality in modalities:
                # Extract 2D slice from 3D volume
                mod_volume = pt_img_dict[modality].get_fdata()
                mod_slice = mod_volume[:, :, slice_idx]
                
                # Also get the segmentation slice
                seg_volume = pt_img_dict['seg'].get_fdata()
                seg_slice = seg_volume[:, :, slice_idx]
                
                # Normalize this 2D slice
                img_normalized, mask_normalized = normalize(mod_slice, seg_slice)
                
                # Convert to numpy and add to channels
                img_np = img_normalized.cpu().numpy()
                channels.append(img_np)
            
            # Stack modalities as channels: (H, W, num_modalities)
            if len(channels) == 1:
                img_multi = np.expand_dims(channels[0], axis=-1)  # (H, W, 1)
            else:
                img_multi = np.stack(channels, axis=-1)  # (H, W, C)
            
            # Mask is same regardless of modality
            mask_np = mask_normalized.cpu().numpy()
            mask_np = np.expand_dims(mask_np, axis=-1)  # (H, W, 1)
            
            all_images.append(img_multi)
            all_masks.append(mask_np)
    
    X = np.array(all_images)
    Y = np.array(all_masks)
    
    print(f"Preprocessing complete! Shape: {X.shape}")
    print(f"Memory usage: ~{X.nbytes / 1e6:.1f} MB (images) + {Y.nbytes / 1e6:.1f} MB (masks)")
    
    return X, Y

In [ ]:
# Now let's preprocess all our data!
# This takes 2-3 minutes but only needs to run once.
# If your kernel crashes, just rerun this cell.

print("=" * 60)
print("PREPROCESSING DATA")
print("=" * 60)

X_train, Y_train = create_numpy_arrays(train_ids)
X_val, Y_val = create_numpy_arrays(val_ids)
X_test, Y_test = create_numpy_arrays(test_ids)

print("\n" + "=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print(f"Training:   {X_train.shape[0]} slices, {X_train.shape[-1]} channel(s)")
print(f"Validation: {X_val.shape[0]} slices, {X_val.shape[-1]} channel(s)")
print(f"Test:       {X_test.shape[0]} slices, {X_test.shape[-1]} channel(s)")
print("=" * 60)

## Step 4: Let's take a look at the performance metrics commonly used in semantic segmentation.

The 2 common ones are
- Dice
- Jaccard/Intersection of Union (IoU)

Both Dice and Jaccard indices are bounded between 0 and 1 with 0 indicating completely inaccurate model prediction and 1 indicating completely accurate model prediction.

Performance metrics in image segmentation: https://towardsdatascience.com/metrics-to-evaluate-your-semantic-segmentation-model-6bcb99639aa2

In [ ]:
smooth = 1.0  # Smoothing factor to avoid division by zero

# Dice Coefficient
def dice_coef(y_true, y_pred):
    """
    Dice coefficient for semantic segmentation.
    
    Dice = (2 * |X ∩ Y|) / (|X| + |Y|)
         = 2*sum(|A*B|) / (sum(A²) + sum(B²))
    
    Args:
        y_true: Ground truth tensor
        y_pred: Predicted tensor (after sigmoid, values in [0,1])
        
    Returns:
        Dice coefficient (higher is better, range [0,1])
    """

    y_true_f = y_true.view(-1)
    y_pred_f = y_pred.view(-1)
    intersection = (y_true_f * y_pred_f).sum()
    return (2. * intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)


# Jaccard Coefficient (IoU)
def jacard_coef(y_true, y_pred):
    """
    Jaccard coefficient for semantic segmentation (also called IoU).
    
    Jaccard = |X ∩ Y| / |X ∪ Y|
            = sum(|A*B|) / (sum(|A|) + sum(|B|) - sum(|A*B|))
    
    Args:
        y_true: Ground truth tensor
        y_pred: Predicted tensor (after sigmoid, values in [0,1])
        
    Returns:
        Jaccard coefficient (higher is better, range [0,1])
    """
    y_true_f = y_true.view(-1)
    y_pred_f = y_pred.view(-1)
    intersection = (y_true_f * y_pred_f).sum()
    return (intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() - intersection + smooth)

## Step 5: Let's take a look at loss functions for semantic segmentation.

The default choice of loss function for segmentation and other classification tasks is Binary Cross-Entropy (BCE). Here since the metric  Dice or Jaccard Coefficient is being used to judge model performance, the loss functions that are derived from these metrics - typically in the form 1 - f(x) where f(x) is the metric in question.

Loss functions in image segmentation: https://medium.com/@junma11/loss-functions-for-medical-image-segmentation-a-taxonomy-cefa5292eec0

In [12]:
# The 2 loss functions we will look at are Dice and Jaccard loss
def dice_coef_loss(y_true, y_pred):
    """
    Dice loss for semantic segmentation.
    Minimizing this loss corresponds to maximizing the Dice coefficient.
    """
    return 1 - dice_coef(y_true, y_pred)

def jaccard_coef_loss(y_true, y_pred):
    """
    Jaccard loss for semantic segmentation.
    Minimizing this loss corresponds to maximizing the Jaccard coefficient.
    """
    return 1 - jaccard_coef(y_true, y_pred)


## Step 6: Now comes the most interesing bit of our learning!

We will now build the Unet model we are going to use for semantic segmentation of WT.

What is a UNet: https://towardsdatascience.com/understanding-semantic-segmentation-with-unet-6be4f42d4b47

Video explaining UNet: https://www.youtube.com/watch?v=azM57JuQpQI


In [ ]:
class UNet(nn.Module):
    """
    U-Net architecture for semantic segmentation.
    
    Architecture:
    - 4 downsampling blocks (encoder) with max pooling
    - Bottleneck (center)
    - 4 upsampling blocks (decoder) with skip connections
    - Final 1x1 conv for classification
    
    Args:
        in_channels: Number of input channels (1 for single modality, 3 for multi-modal)
        out_channels: Number of output channels (1 for binary, 4 for multi-class)
    """
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def conv_block(in_channels, out_channels):
            """Double convolution block: Conv-ReLU-BN-Conv-ReLU-BN"""
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.BatchNorm2d(out_channels),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.BatchNorm2d(out_channels)
            )

        # Encoder (downsampling path)
        self.down1 = conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down4 = conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.center = conv_block(512, 1024)

        # Decoder (upsampling path with skip connections)
        self.up4 = conv_block(1024 + 512, 512)
        self.up3 = conv_block(512 + 256, 256)
        self.up2 = conv_block(256 + 128, 128)
        self.up1 = conv_block(128 + 64, 64)

        # Final classification layer
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        down1 = self.down1(x)
        pool1 = self.pool1(down1)

        down2 = self.down2(pool1)
        pool2 = self.pool2(down2)

        down3 = self.down3(pool2)
        pool3 = self.pool3(down3)

        down4 = self.down4(pool3)
        pool4 = self.pool4(down4)

        # Bottleneck
        center = self.center(pool4)

        # Decoder with skip connections
        up4 = F.interpolate(center, scale_factor=2, mode='bilinear', align_corners=True)
        up4 = torch.cat([up4, down4], dim=1)
        up4 = self.up4(up4)

        up3 = F.interpolate(up4, scale_factor=2, mode='bilinear', align_corners=True)
        up3 = torch.cat([up3, down3], dim=1)
        up3 = self.up3(up3)

        up2 = F.interpolate(up3, scale_factor=2, mode='bilinear', align_corners=True)
        up2 = torch.cat([up2, down2], dim=1)
        up2 = self.up2(up2)

        up1 = F.interpolate(up2, scale_factor=2, mode='bilinear', align_corners=True)
        up1 = torch.cat([up1, down1], dim=1)
        up1 = self.up1(up1)

        # Final classification with sigmoid activation
        final = self.final(up1)
        return torch.sigmoid(final)

Instead of constant learning rate, how to use a gradually decreasing learning rate instead: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

Lets visualize the model now!

In [ ]:
from torchviz import make_dot

model = UNet()

# Create a random tensor with the same size as your input
x = torch.randn(1, 1, 128, 128)  # Example input size

# Forward pass to generate the graph
y = model(x)

# Use make_dot to create a visualization
model_graph = make_dot(y, params=dict(model.named_parameters()))

# Save or display the graph
model_graph.format = 'png'
model_graph.render('unet_model')

# Display the graph in the notebook (if using Jupyter)
display(model_graph)

In [ ]:
plt.imshow(y.detach().numpy()[0, 0, :,:])

## Step 7: Creating the DataLoader & Training

Now that we have preprocessed numpy arrays, creating a dataset is simple! PyTorch's `TensorDataset` wraps our arrays and the `DataLoader` handles batching and shuffling.

**Key parameters:**
- `batch_size`: Number of samples per batch (64 is a good balance)
- `shuffle=True`: Randomize order each epoch (prevents memorization)
- `num_workers`: Parallel data loading (0 for main thread, 2-4 for speed)

In [ ]:
# Convert numpy arrays to PyTorch tensors and create datasets
# Note: We transpose from (N, H, W, C) to (N, C, H, W) - PyTorch convention
train_dataset = TensorDataset(
    torch.from_numpy(X_train).permute(0, 3, 1, 2).float(),
    torch.from_numpy(Y_train).permute(0, 3, 1, 2).float()
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val).permute(0, 3, 1, 2).float(),
    torch.from_numpy(Y_val).permute(0, 3, 1, 2).float()
)

test_dataset = TensorDataset(
    torch.from_numpy(X_test).permute(0, 3, 1, 2).float(),
    torch.from_numpy(Y_test).permute(0, 3, 1, 2).float()
)

# Create data loaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"DataLoaders created successfully!")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Now we'll use our preprocessed arrays to train the model!

In [ ]:
# Let's confirm the size of the training, validation and test datasets
print(f'There are {len(train_dataset)} slices in training set')
print(f'There are {len(val_dataset)} slices in validation set')
print(f'There are {len(test_dataset)} slices in test set')
print(f'\nNote: Each "slice" is a preprocessed 2D image-mask pair')

There are 236 images in training set
There are 59 images in validation set
There are 74 images in test set


### 7.1. Training the Baseline Model

We will now train our first model. This will serve as our **baseline**. It uses:
- **Input:** A single, middle T2 slice per patient.
- **Loss Function:** Binary Cross-Entropy (`nn.BCELoss`).
- **Output:** A binary mask for the whole tumor.

All experiments in Step 9 will be compared against this baseline's performance.

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.0001, use_gpu=True):
    """
    Train the U-Net model for brain tumor segmentation.
    
    Args:
        model: The U-Net model to train
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Number of training epochs
        learning_rate: Learning rate for Adam optimizer
        use_gpu: Whether to use GPU if available
    
    Returns:
        DataFrame containing training metrics (epoch, train_loss, val_loss)
    """
    # Setup device
    device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # Loss function: Binary Cross-Entropy (model outputs sigmoid, so use BCELoss)
    criterion = nn.BCELoss()
    
    # Optimizer: Adam with specified learning rate
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print("=" * 60)
    print(f"Training on: {device}")
    print(f"Epochs: {num_epochs} | Learning Rate: {learning_rate} | Batch Size: {train_loader.batch_size}")
    print("=" * 60)
    
    # Storage for metrics
    metrics = {'epoch': [], 'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        # ==================== TRAINING PHASE ====================
        model.train()
        running_loss = 0.0
        
        for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            images, masks = images.to(device), masks.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            
            # Compute loss
            loss = criterion(outputs, masks)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        train_loss = running_loss / len(train_loader)
        
        # ==================== VALIDATION PHASE ====================
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        # Record metrics
        metrics['epoch'].append(epoch + 1)
        metrics['train_loss'].append(train_loss)
        metrics['val_loss'].append(val_loss)
        
        # Print progress
        print(f"Epoch {epoch+1:2d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    print("=" * 60)
    print("Training complete!")
    print("=" * 60)
    
    return pd.DataFrame(metrics)

In [ ]:
# Initialize the model
# in_channels=1 because we're using single modality (T2)
# For Q5 (multi-modal), you'll change this to in_channels=3 or more
model = UNet(in_channels=1, out_channels=1)

# Train the model
metrics_df = train_model(model, train_loader, val_loader, num_epochs=20, use_gpu=True)

# Display final metrics
print("\nFinal Metrics:")
print(metrics_df.tail())

### 7.2. Saving the Baseline Model (Optional but Recommended)

Training takes time! To avoid retraining the model every time you work on this notebook, you can save its learned parameters (weights). I know how frustrating it is when the session disconnects/times out.

In [ ]:
# Define a path to save the model
MODEL_PATH = './baseline_unet_t2.pth'

# Save the model's state dictionary
torch.save(model.state_dict(), MODEL_PATH)

print(f"Baseline model saved to {MODEL_PATH}")

# You can later load it back like this:
# model_to_load = UNet(in_channels=1, out_channels=1)
# model_to_load.load_state_dict(torch.load(MODEL_PATH))
# model_to_load.to(device) # Don't forget to move it to the GPU!
# print("Baseline model reloaded successfully!")

## Step 8: Your turn

### 8.1. Learning curves

Learning curves are a widely used diagnostic tool in machine learning for algorithms that learn from a training dataset incrementally. The model can be evaluated on the training dataset and on a hold out validation dataset after each update during training and plots of the measured performance can created to show learning curves.

Importance of learning curves: https://towardsdatascience.com/learning-curve-to-identify-overfitting-underfitting-problems-133177f38df5

Below, figure out how to visualize a learning curve using the metrics that you've created above.

In [ ]:
# ============================================================
# TODO: Visualize Learning Curves
# ============================================================

# Create a plot showing training and validation loss over epochs
plt.figure(figsize=(10, 6))

# TODO: Plot training loss
plt.plot(None, None, label='Training Loss', marker='o')

# TODO: Plot validation loss  
plt.plot(None, None, label='Validation Loss', marker='o')

# TODO: Add labels and title
plt.xlabel(None)
plt.ylabel(None)
plt.title(None)

plt.legend()
plt.grid(True)
plt.show()

# Questions to consider after plotting:
# - Is the model overfitting (val_loss >> train_loss)?
# - Has the model converged or does it need more epochs?
# - At which epoch does the best validation performance occur?

Understanding difference between batch and epoch in deep learning: https://machinelearningmastery.com/difference-between-a-batch-and-an-epoch/

Are there ways to automate hyper-parameter tuning: https://neptune.ai/blog/hyperparameter-tuning-in-python-complete-guide

### 8.2 Visualizing the outputs
Figure out a way to visualize the outputs of the model. Take a look at the ground truth masks next to the outputs of the model for a few images. Try to look at a slice in the volume that matters. Would you say the model is doing a good or bad job at creating predictions right now?

*Hint*: Remember that `matplotlib` and `numpy` work on the CPU. Before you can visualize a tensor, you must first move it from the GPU back to the CPU using `.cpu()` (e.g., `tensor.cpu().numpy()`).

In [ ]:
# ============================================================
# TODO: Visualize Model Predictions vs Ground Truth
# ============================================================

# Set model to evaluation mode
model.eval()
device = next(model.parameters()).device

# Get one batch from test set
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        
        # TODO: Get model predictions
        outputs = None  # model(...)
        
        # Visualize first 3 samples
        num_samples = min(3, images.shape[0])
        
        for i in range(num_samples):
            plt.figure(figsize=(15, 5))
            
            # Input Image
            plt.subplot(1, 3, 1)
            # TODO: Extract the i-th image from the batch (hint: index and squeeze channel dim)
            input_img = None
            plt.imshow(input_img, cmap='gray')
            plt.title('Input Image')
            plt.axis('off')
            
            # Ground Truth Mask
            plt.subplot(1, 3, 2)
            # TODO: Extract the i-th mask from the batch
            true_mask = None 
            plt.imshow(true_mask, cmap='gray')
            plt.title('Ground Truth')
            plt.axis('off')
            
            # Predicted Mask
            plt.subplot(1, 3, 3)
            # TODO: Extract prediction, threshold at 0.5, convert to numpy
            pred_mask = None
            plt.imshow(pred_mask, cmap='gray')
            plt.title('Prediction')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
        
        break  # Only process one batch

# Analysis questions:
# - Does the model capture tumor boundaries accurately?
# - Are there false positives (predicting tumor where there's none)?
# - Are there false negatives (missing actual tumor regions)?
# - Which cases does the model handle well vs poorly?

## Step 9: Extensions and Experiments

Now that you have a working brain tumor segmentation model, it's time to experiment! The questions below will help you understand what factors affect model performance. 

**Important**: For each question, **copy the relevant cells from above and modify them** - don't change your original working code.

---

### Question 1: Jaccard Loss Function [2 points]

**Task**: Retrain the model using **Jaccard loss** instead of BCE loss, and track the **Jaccard coefficient** as a metric.

**Hints**:
- Look at the loss functions defined in Step 5
- You'll need to modify the `train_model` function's `criterion` variable
- Consider adding a metrics tracking loop to compute Jaccard score each epoch

**Report**: 
- Final Jaccard scores on training and validation sets
- Does Jaccard loss lead to better Jaccard scores than BCE loss?



In [ ]:
# ============================================================
# Question 1: Train with Jaccard Loss
# ============================================================

# TODO: Modify the train_model function to use Jaccard loss

def train_model_q1(model, train_loader, val_loader, num_epochs=10, learning_rate=0.0001, use_gpu=True):
    device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # TODO: Replace with appropriate loss function from Step 5
    criterion = None
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    metrics = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_jaccard': [], 'val_jaccard': []}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_jaccard = 0.0
        
        for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # TODO: Compute Jaccard coefficient for this batch
            with torch.no_grad():
                batch_jaccard = None  # jacard_coef(...)
                running_jaccard += batch_jaccard.item()
        
        train_loss = running_loss / len(train_loader)
        train_jaccard = running_jaccard / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_jaccard = 0.0
        
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                
                # TODO: Compute validation loss and Jaccard
                loss = None
                jaccard = None
                
                val_loss += loss.item()
                val_jaccard += jaccard.item()
        
        val_loss /= len(val_loader)
        val_jaccard /= len(val_loader)
        
        metrics['epoch'].append(epoch + 1)
        metrics['train_loss'].append(train_loss)
        metrics['val_loss'].append(val_loss)
        metrics['train_jaccard'].append(train_jaccard)
        metrics['val_jaccard'].append(val_jaccard)
        
        print(f"Epoch {epoch+1:2d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Jaccard: {val_jaccard:.4f}")
    
    return pd.DataFrame(metrics)

# TODO: Train the model
model_q1 = UNet(None, None)  # Adjust in_channels and out_channels as needed
metrics_q1 = None  # train_model_q1(...)

### Question 2: Tracking Multiple Metrics [2 points]

**Task**: Keep using BCE loss, but track **both Dice and Jaccard coefficients** as additional metrics during training.

**Hints**:
- The metric functions are already defined - you just need to call them
- Add new columns to the metrics dictionary: 'train_dice', 'val_dice', 'train_jaccard', 'val_jaccard'
- Compute these during the training and validation loops (inside `torch.no_grad()` for validation)

**Report**:
- Create 3 plots:
  - Plot 1: Training loss vs Validation loss
  - Plot 2: Training Dice vs Validation Dice
  - Plot 3: Training Jaccard vs Validation Jaccard
- Which metric stabilizes faster?
- How closely do Dice and Jaccard track each other?


In [ ]:
# ============================================================
# Question 2: Track Dice and Jaccard with BCE Loss
# ============================================================

def train_model_q2(model, train_loader, val_loader, num_epochs=10, learning_rate=0.0001, use_gpu=True):
    device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.BCELoss()  # Keep BCE loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # TODO: Add Dice and Jaccard to metrics dictionary
    metrics = {'epoch': [], 'train_loss': [], 'val_loss': []}  # Add more keys here
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_dice = 0.0
        running_jaccard = 0.0
        
        for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # TODO: Compute and accumulate Dice and Jaccard scores
            with torch.no_grad():
                batch_dice = None
                batch_jaccard = None
                running_dice += None
                running_jaccard += None
        
        # TODO: Compute averages for all metrics
        train_loss = None
        train_dice = None
        train_jaccard = None
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_dice = 0.0
        val_jaccard = 0.0
        
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                
                # TODO: Compute all three validation metrics
                pass
        
        # TODO: Average validation metrics
        val_loss /= len(val_loader)
        val_dice = None
        val_jaccard = None
        
        # TODO: Store all metrics
        metrics['epoch'].append(epoch + 1)
        # ... add train_loss, val_loss, train_dice, val_dice, train_jaccard, val_jaccard
        
        print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f}/{val_loss:.4f} | Dice: {train_dice:.4f}/{val_dice:.4f} | Jaccard: {train_jaccard:.4f}/{val_jaccard:.4f}")
    
    return pd.DataFrame(metrics)

# TODO: Create 3 subplots to visualize loss, Dice, and Jaccard curves
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# ... plotting code ...

---

### Question 3: Different MRI Modality [2 points]

**Task**: Train a model using **a different MRI modality** (T1, T1CE, or FLAIR) instead of T2.

**Hints**:
- The `create_numpy_arrays()` function has a `modalities` parameter
- You'll need to rerun preprocessing with your chosen modality
- Train a new model and compare to the T2 baseline

**Report**:
- Which modality did you choose and what was its validation Dice score?
- Compare to T2 baseline - which performs better?
- Visualize a few predictions - what differences do you notice in how it segments tumors?


In [ ]:
# ============================================================
# Question 3: Try a Different Modality
# ============================================================

# TODO: Preprocess with a different modality (choose: 't1', 't1ce', or 'flair')
# X_train_q3, Y_train_q3 = create_numpy_arrays(train_ids, modalities=[None])
# X_val_q3, Y_val_q3 = create_numpy_arrays(val_ids, modalities=[None])
# X_test_q3, Y_test_q3 = create_numpy_arrays(test_ids, modalities=[None])

# TODO: Create datasets and loaders
# train_dataset_q3 = TensorDataset(...)
# ...

# TODO: Train model
# model_q3 = UNet(in_channels=1, out_channels=1)
# metrics_q3 = train_model(model_q3, train_loader_q3, val_loader_q3, num_epochs=20, use_gpu=True)

# TODO: Visualize a few predictions and compare to T2 baseline

---

### Question 4: Using Multiple Slices [3 points]

**Task**: Instead of using only the middle slice, use **three slices** per patient to increase training data.

**Hints**:
- Look at the `slice_indices` parameter in `create_numpy_arrays()`
- Currently it's `[lambda d: d//2]` (one slice at 50% depth)
- Think about how to specify multiple slice positions (e.g., 25%, 50%, 75%)
- After preprocessing with more slices, your dataset will be 3x larger

**Report**:
- Does having 3x more training data improve validation Dice score?
- Compare learning curves between single-slice and multi-slice training

**Note**: This approach treats each slice independently. For capturing spatial context between slices, you'd need 3D convolutions or channel stacking (see Q5).



In [ ]:
# ============================================================
# Question 4: Train with Multiple Slices
# ============================================================

# TODO: Modify slice_indices to extract 3 slices instead of 1
# Hint: Use lambda functions to specify slice positions as fractions of depth
# X_train_q4, Y_train_q4 = create_numpy_arrays(
#     train_ids, 
#     slice_indices=[None, None, None]  # e.g., [lambda d: d//4, ...]
# )
# X_val_q4, Y_val_q4 = create_numpy_arrays(val_ids, slice_indices=[None, None, None])
# X_test_q4, Y_test_q4 = create_numpy_arrays(test_ids, slice_indices=[None, None, None])

# TODO: Create datasets and loaders
# train_dataset_q4 = TensorDataset(...)
# ...

# TODO: Train model
# model_q4 = UNet(in_channels=1, out_channels=1)
# metrics_q4 = train_model(model_q4, train_loader_q4, val_loader_q4, num_epochs=20, use_gpu=True)

# TODO: Compare learning curves with single-slice baseline
# fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# ax.plot(metrics_df['epoch'], metrics_df['val_loss'], label='Single Slice')
# ax.plot(None, None, label='Three Slices')  # Plot Q4 validation loss
# ...

---

### Question 5: Multi-Modal Input [4 points]

**Task**: Use **multiple MRI modalities** as different input channels (like RGB images have 3 color channels).

**Hints**:
- Use the `modalities` parameter in `create_numpy_arrays()` to load multiple sequences
- Your preprocessed arrays will have shape (N, H, W, C) where C = number of modalities
- The UNet class has an `in_channels` parameter - does it need to change?
- Think about what value `in_channels` should be when you have multiple modalities

**Requirements**:
- Try at least **3 different combinations** of modalities
- Examples: ['t1', 't2'], ['t2', 'flair'], ['t1', 't1ce', 'flair'], ['t1', 't2', 'flair', 't1ce']

**Report**:
- Table showing each combination and its validation Dice score
- Which combination works best?
- Does using more modalities always help? Why or why not?
- Visualize predictions from your best combination vs single modality

**Reasoning**: Each modality shows different tissue properties. T1 shows anatomy, T2 shows edema/fluid, T1CE shows blood-brain barrier breakdown, FLAIR suppresses CSF. Combining them provides complementary information.

In [ ]:
# ============================================================
# Question 5: Multi-Modal Input
# ============================================================

# TODO: Try at least 3 different modality combinations

# Combination 1: 
# X_train_q5a, Y_train_q5a = create_numpy_arrays(train_ids, modalities=[None, None])
# ...

# Combination 2:
# X_train_q5b, Y_train_q5b = create_numpy_arrays(train_ids, modalities=[None, None, None])
# ...

# Combination 3:
# X_train_q5c, Y_train_q5c = create_numpy_arrays(train_ids, modalities=[None, None, None, None])
# ...

# TODO: For each combination, create datasets
# train_dataset_q5a = TensorDataset(...)

# TODO: Initialize model with correct number of input channels
# model_q5a = UNet(in_channels=None, out_channels=1)  # Match number of modalities

# TODO: Train and record results
# metrics_q5a = train_model(model_q5a, ...)

# TODO: Create comparison table
# results_q5 = pd.DataFrame({
#     'Combination': ['...', '...', '...'],
#     'Val_Dice': [None, None, None]
# })
# print(results_q5)

---

### Question 6: Multi-Class Segmentation [5 points]

**Task**: Instead of binary segmentation (tumor vs background), predict **all 4 regions**: background (0), necrotic core (1), edema (2), enhancing tumor (4).

**Hints**:
- You need to modify the `normalize()` function to preserve class labels (don't convert to binary)
- The class labels are 0, 1, 2, 4 - but PyTorch CrossEntropyLoss expects 0, 1, 2, 3 (remap 4→3)
- Change `out_channels` in UNet to the number of classes
- Use `nn.CrossEntropyLoss()` which expects logits (no sigmoid)
- Remove the `torch.sigmoid()` from UNet's `forward()` method
- For predictions, use `torch.argmax(outputs, dim=1)` to get the predicted class

**Visualization hint**:
```python
# Color-coded visualization (each class gets a different color)
plt.imshow(pred_class, cmap='tab10')
plt.colorbar(label='Class: 0=BG, 1=Necrotic, 2=Edema, 3=Enhancing')
```

**Report**:

- Compute Dice score for each class separately (background, necrotic, edema, enhancing)
- Which tumor region is easiest/hardest to segment?
- Visualize 3-5 predictions with color-coded classes
- Compare multi-class performance to binary (whole tumor) - what's the tradeoff?

**Warning**: Multi-class is significantly harder - expect lower per-class scores than binary segmentation.



In [ ]:
# ============================================================
# Question 6: Multi-Class Segmentation
# ============================================================

# TODO: Modify normalize function to preserve class labels
def normalize_multiclass(input_image, input_mask, percentile=0.001, eps=1e-6):
    # Image normalization (same as before)
    if not isinstance(input_image, torch.Tensor):
        input_image = torch.tensor(input_image, dtype=torch.float32)
    
    img_array = input_image.contiguous().view(-1)
    min_img = torch.quantile(img_array, percentile)
    max_img = torch.quantile(img_array, 1 - percentile)
    img_normalized = (input_image - min_img) / (max_img - min_img + eps)
    img_normalized = torch.clamp(img_normalized, 0, 1).float()
    
    # TODO: Mask preprocessing - preserve classes but remap 4→3
    if not isinstance(input_mask, torch.Tensor):
        input_mask = torch.tensor(input_mask, dtype=torch.int32)
    
    mask_multiclass = None  # Clone input_mask and remap label 4 to 3
    
    return img_normalized, mask_multiclass

# TODO: Create preprocessing function for multi-class (copy and modify create_numpy_arrays)
def create_numpy_arrays_multiclass(patient_ids, modalities=['t2'], slice_indices=[lambda d: d//2], data_path=TRAIN_DATA_PATH):
    all_images = []
    all_masks = []
    
    for patient_id in tqdm(patient_ids, desc="Loading patients"):
        niftis_to_load = modalities + ['seg']
        pt_img_dict = read_patient_niftis(patient_id, niftis_to_load=niftis_to_load, data_path=data_path)
        
        depth = pt_img_dict['seg'].shape[2]
        
        for slice_fn in slice_indices:
            slice_idx = slice_fn(depth)
            channels = []
            
            for modality in modalities:
                mod_slice = pt_img_dict[modality].get_fdata()[:, :, slice_idx]
                seg_slice = pt_img_dict['seg'].get_fdata()[:, :, slice_idx]
                
                # TODO: Use normalize_multiclass instead of normalize
                img_normalized, mask_normalized = None, None
                
                img_np = img_normalized.cpu().numpy()
                channels.append(img_np)
            
            if len(channels) == 1:
                img_multi = np.expand_dims(channels[0], axis=-1)
            else:
                img_multi = np.stack(channels, axis=-1)
            
            mask_np = mask_normalized.cpu().numpy()
            mask_np = np.expand_dims(mask_np, axis=-1)
            
            all_images.append(img_multi)
            all_masks.append(mask_np)
    
    return np.array(all_images), np.array(all_masks)

# TODO: Preprocess data
# X_train_q6, Y_train_q6 = create_numpy_arrays_multiclass(train_ids)
# ...

# TODO: Modify UNet class - remove sigmoid from forward() method
class UNetMultiClass(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super(UNetMultiClass, self).__init__()
        # Copy UNet architecture from above
        pass
    
    def forward(self, x):
        # Copy forward pass from above but remove sigmoid
        # TODO: Return logits (no activation)
        pass

# TODO: Train with CrossEntropyLoss
model_q6 = UNetMultiClass(None, None)  # Adjust in_channels and out_channels (4 for multi-class)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_q6.to(device)
criterion_q6 = None  # Use CrossEntropyLoss
optimizer_q6 = optim.Adam(model_q6.parameters(), lr=0.0001)

# Training loop (modify to handle multi-class)
for epoch in range(num_epochs):
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model_q6(images)
        
        # TODO: Prepare masks for CrossEntropyLoss (remove channel dim, convert to long)
        masks_prepared = None  # masks.squeeze(1).long()
        
#         loss = criterion_q6(outputs, masks_prepared)
#         ...

# TODO: Visualization with color-coding
# with torch.no_grad():
#     for images, masks in test_loader:
#         images = images.to(device)
#         outputs = model_q6(images)
#         
#         # TODO: Get predicted class (use argmax)
#         pred_class = None  # torch.argmax(outputs, dim=1)
#         
#         # Visualize
#         plt.figure(figsize=(15, 5))
#         plt.subplot(1, 3, 1)
#         plt.imshow(images[0, 0].cpu(), cmap='gray')
#         plt.title('Input')
#         
#         plt.subplot(1, 3, 2)
#         plt.imshow(masks[0, 0].cpu(), cmap='tab10')
#         plt.colorbar(label='True Class')
#         plt.title('Ground Truth')
#         
#         plt.subplot(1, 3, 3)
#         # TODO: Visualize predicted classes with colors
#         plt.imshow(None, cmap='tab10')
#         plt.colorbar(label='Predicted Class')
#         plt.title('Prediction')
#         plt.show()
#         break

# TODO: Compute per-class Dice scores
# def dice_per_class(y_true, y_pred, num_classes=4):
#     dice_scores = []
#     for c in range(num_classes):
#         # TODO: Compute Dice for class c
#         true_c = None  # (y_true == c).float()
#         pred_c = None  # (y_pred == c).float()
#         dice_c = None  # dice_coef(true_c, pred_c)
#         dice_scores.append(dice_c.item())
#     return dice_scores

---

### Question 7: Reflection [2 points]

Write a brief reflection addressing:

1. **Learning**: Key insights about semantic segmentation and medical imaging
2. **Challenges**: Most difficult aspects of the experiments

---

## Additional Resources

- Batch vs Epoch: https://machinelearningmastery.com/difference-between-a-batch-and-an-epoch/

- Automating hyperparameter tuning: https://neptune.ai/blog/hyperparameter-tuning-in-python-complete-guide

- Why is image normalizaton needed: https://arthurdouillard.com/post/normalization/

Good luck with your learning!